NeuN-low cutoffs for the sensitivity analysis. Here, NeuN-low is recomputed at 5-25% of WT Rbfox3 expression, and ECs proximal to NeuN-low neurons are re-labeled at the same distance cutoffs (18-103 µm). Output `lowneun_variants.parquet` is read by 07 (`low_pct`).

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
from scipy.spatial import cKDTree

In [ ]:
base = Path("/dss/dssfs03/pn52re/pn52re-dss-0001/cellseg-benchmark/analysis/foxf2/Cellpose_1_nuclei_model")
out_dir = base / "dea" / "sensitivity"
out_dir.mkdir(parents=True, exist_ok=True)
pcts = [5, 10, 15, 20, 25]
cutoffs = [18, 31, 58, 103]  # um, ~10/25/50/75th percentiles of EC-to-NeuN-low distances at p10 (04), kept fixed

In [ ]:
adata = sc.read_h5ad(base / "adatas" / "20260302_temp-after_ec_proximal_to_lowNeuN.h5ad.gz")
obs = adata.obs
xy = np.asarray(adata.obsm["spatial_microns"])[:, :2]
glut = obs["Rbfox3_low"].notna()
expr = sc.get.obs_df(adata, keys=["Rbfox3"], layer="volume_log1p_norm")["Rbfox3"]
wt_expr = expr[glut & (obs["genotype"] == "WT")]
is_ec = (obs["cell_type"] == "ECs").to_numpy()
is_glut = (obs["cell_type"] == "Neurons-Glut").to_numpy()

In [ ]:
cols, dists = {}, {}
for p in pcts:
    low = glut & (expr < wt_expr.quantile(p / 100))
    cols[f"p{p}|Rbfox3_low"] = low.where(glut)
    src_all = low.to_numpy() & is_glut
    d = pd.Series(np.nan, index=obs.index)
    for s in obs["sample2"].unique():
        m = (obs["sample2"] == s).to_numpy()
        src, tgt = m & src_all, m & is_ec
        if src.any() and tgt.any():
            d.loc[tgt] = cKDTree(xy[src]).query(xy[tgt])[0]
    dists[p] = d

In [ ]:
prox_cols = [f"ec_proximal_to_lowNeuN_{d}um" for d in cutoffs]
for p in pcts:
    for c, d in zip(prox_cols, cutoffs):
        cols[f"p{p}|{c}"] = dists[p] <= d
out = pd.DataFrame(cols)

In [ ]:
out.to_parquet(out_dir / "lowneun_variants.parquet")

In [ ]:
pct_low = pd.DataFrame({
    p: out.loc[glut, f"p{p}|Rbfox3_low"].eq(True).groupby(obs.loc[glut, "sample2"], observed=True).mean().mul(100)
    for p in pcts
})
n_prox = pd.DataFrame({
    p: out.loc[is_ec, [f"p{p}|{c}" for c in prox_cols]].groupby(obs.loc[is_ec, "genotype"], observed=True).sum().set_axis(prox_cols, axis=1).stack()
    for p in pcts
})
pct_low.round(1).to_csv(out_dir / "lowneun_pct_per_sample.csv")
n_prox.to_csv(out_dir / "n_proximal_ECs.csv")
display(pct_low.round(1), n_prox)